# COE 292 – AI Final Project
## Maternal Health Risk Classification

**Team ID:** [Enter Team Number]  
**Team Members:** [Name 1, Name 2, Name 3]  
**Dataset:** Maternal Health Risk – UCI ML Repository  
**Source:** https://archive.ics.uci.edu/dataset/863/maternal+health+risk

## 0. Install & Import Libraries

In [ ]:
# Install ucimlrepo to download the dataset directly from UCI
!pip install ucimlrepo -q

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from ucimlrepo import fetch_ucirepo

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import StratifiedKFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (confusion_matrix, accuracy_score,
                             precision_score, recall_score, f1_score)

# Set a consistent random seed
SEED = 42
np.random.seed(SEED)

print('All libraries imported successfully.')

---
## 1. Dataset Exploration and Preparation
### 1.1 Load Dataset from the Internet

In [ ]:
# Fetch the Maternal Health Risk dataset directly from UCI ML Repository
maternal_health_risk = fetch_ucirepo(id=863)

# Get features and target
X = maternal_health_risk.data.features
y = maternal_health_risk.data.targets

# Combine into one DataFrame for EDA
df = X.copy()
df['RiskLevel'] = y.values.ravel()

print('Dataset loaded successfully!')
print(f'Shape: {df.shape}')
print(df.head())

### 1.2 Dataset Background

In [ ]:
# Print metadata from the UCI repository
print('=== Dataset Metadata ===')
print(maternal_health_risk.metadata)
print()
print('=== Variable Information ===')
print(maternal_health_risk.variables)

In [ ]:
# Dataset summary
num_samples = df.shape[0]
num_features = X.shape[1]
num_classes = df['RiskLevel'].nunique()
class_dist = df['RiskLevel'].value_counts()

print('=== Dataset Summary ===')
print(f'Source          : UCI Machine Learning Repository (ID: 863)')
print(f'Purpose         : Predict maternal health risk level (low/mid/high) from IoT sensor data')
print(f'Number of samples  : {num_samples}')
print(f'Number of features : {num_features}')
print(f'Number of classes  : {num_classes}')
print()
print('Class Distribution:')
for cls, count in class_dist.items():
    print(f'  {cls:15s}: {count} samples ({100*count/num_samples:.1f}%)')

### 1.3 Exploratory Data Analysis (EDA)

In [ ]:
# Check for missing values
print('=== Missing Values ===')
print(df.isnull().sum())
print(f'\nTotal missing values: {df.isnull().sum().sum()}')

In [ ]:
# Descriptive statistics
print('=== Descriptive Statistics ===')
print(df.describe())

In [ ]:
# Feature names
feature_names = list(X.columns)
print('Features:', feature_names)

### 1.4 Data Visualization – Scatter Plots

In [ ]:
# Define colors and markers for each risk class
class_labels = ['low risk', 'mid risk', 'high risk']
colors = {'low risk': 'green', 'mid risk': 'orange', 'high risk': 'red'}
markers = {'low risk': 'o', 'mid risk': 's', 'high risk': '^'}

# Helper: draw one scatter plot
def scatter_plot(ax, x_col, y_col, df):
    for label in class_labels:
        subset = df[df['RiskLevel'] == label]
        ax.scatter(subset[x_col], subset[y_col],
                   c=colors[label], marker=markers[label],
                   label=label, alpha=0.6, edgecolors='k', linewidths=0.3, s=30)
    ax.set_xlabel(x_col, fontsize=11)
    ax.set_ylabel(y_col, fontsize=11)
    ax.set_title(f'{x_col} vs {y_col}', fontsize=12, fontweight='bold')
    ax.legend(title='Risk Level', fontsize=9)
    ax.grid(True, linestyle='--', alpha=0.4)

# 4 scatter plots
fig, axes = plt.subplots(2, 2, figsize=(13, 10))

scatter_plot(axes[0, 0], 'Age',            'SystolicBP',    df)
scatter_plot(axes[0, 1], 'SystolicBP',     'DiastolicBP',   df)
scatter_plot(axes[1, 0], 'BS',             'BodyTemp',      df)
scatter_plot(axes[1, 1], 'HeartRate',      'SystolicBP',    df)

fig.suptitle('Maternal Health Risk – Scatter Plots', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

### 1.5 Preprocessing & Cross-Validation Setup

In [ ]:
# Encode target labels to integers (required by sklearn metrics)
le = LabelEncoder()
y_encoded = le.fit_transform(df['RiskLevel'])  # high=0, low=1, mid=2
X_array = X.values

print('Class encoding:', dict(zip(le.classes_, le.transform(le.classes_))))

# Set up 5-fold stratified cross-validation
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
print('\n5-Fold Stratified Cross-Validation configured.')
print('Each fold maintains the original class distribution.')

### Helper Function – Compute All Metrics
> This function computes all required metrics for one fold given true and predicted labels.

In [ ]:
# Compute accuracy, macro precision, recall, F1, and
# macro sensitivity/specificity from the confusion matrix.
def compute_metrics(y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average='macro', zero_division=0)
    rec  = recall_score(y_true, y_pred, average='macro', zero_division=0)
    f1   = f1_score(y_true, y_pred, average='macro', zero_division=0)

    # Per-class sensitivity (recall) and specificity from confusion matrix
    cm = confusion_matrix(y_true, y_pred)
    n_classes = cm.shape[0]
    sensitivities = []
    specificities = []
    for i in range(n_classes):
        TP = cm[i, i]
        FN = cm[i, :].sum() - TP
        FP = cm[:, i].sum() - TP
        TN = cm.sum() - TP - FN - FP
        sens = TP / (TP + FN) if (TP + FN) > 0 else 0.0
        spec = TN / (TN + FP) if (TN + FP) > 0 else 0.0
        sensitivities.append(sens)
        specificities.append(spec)

    macro_sens = np.mean(sensitivities)
    macro_spec = np.mean(specificities)

    return {
        'Accuracy': acc,
        'Sensitivity': macro_sens,
        'Specificity': macro_spec,
        'Precision': prec,
        'Recall': rec,
        'F1-Score': f1
    }

# Display per-fold results table and mean ± std
def show_results_table(fold_results, model_name):
    results_df = pd.DataFrame(fold_results)
    results_df.index = [f'Fold {i+1}' for i in range(len(fold_results))]

    mean_row = results_df.mean()
    std_row  = results_df.std()
    summary_row = mean_row.map('{:.4f}'.format) + ' ± ' + std_row.map('{:.4f}'.format)

    display_df = results_df.round(4).copy()
    display_df.loc['Mean ± Std'] = summary_row

    print(f'\n=== {model_name} – Per-Fold Results ===')
    print(display_df.to_string())
    return results_df

# Bar chart of average metrics
def plot_avg_metrics(results_df, model_name):
    avg = results_df.mean()
    fig, ax = plt.subplots(figsize=(8, 4))
    bars = ax.bar(avg.index, avg.values, color=['steelblue','tomato','mediumseagreen',
                                                 'gold','darkorange','mediumpurple'],
                  edgecolor='black', width=0.55)
    for bar, val in zip(bars, avg.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{val:.3f}', ha='center', va='bottom', fontsize=9)
    ax.set_ylim(0, 1.12)
    ax.set_ylabel('Score', fontsize=11)
    ax.set_title(f'{model_name} – Average Performance Metrics (5-Fold CV)', fontsize=12, fontweight='bold')
    ax.grid(axis='y', linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.show()

print('Helper functions defined.')

---
## 2. Classification Algorithm 1 – K-Nearest Neighbors (KNN)

### 2.1 Why Scaling is Needed for KNN

KNN classifies a new sample by computing its **Euclidean distance** to all training samples:

$$d(p, q) = \sqrt{\sum_{i=1}^{n}(p_i - q_i)^2}$$

Features with large numeric ranges (e.g., SystolicBP 70–160) will dominate the distance calculation compared to features with small ranges (e.g., BodyTemp 98–103).  
**StandardScaler** normalizes each feature to mean=0 and std=1, so all features contribute equally to the distance.

### 2.2 Choose Optimal K

In [ ]:
# Test K values from 1 to 25 using 5-fold CV
k_values = list(range(1, 26))
k_accuracies = []

for k in k_values:
    fold_accs = []
    for train_idx, test_idx in skf.split(X_array, y_encoded):
        X_train, X_test = X_array[train_idx], X_array[test_idx]
        y_train, y_test = y_encoded[train_idx], y_encoded[test_idx]

        # Scale inside the fold to avoid data leakage
        scaler = StandardScaler()
        X_train_sc = scaler.fit_transform(X_train)
        X_test_sc  = scaler.transform(X_test)

        knn = KNeighborsClassifier(n_neighbors=k)
        knn.fit(X_train_sc, y_train)
        fold_accs.append(accuracy_score(y_test, knn.predict(X_test_sc)))

    k_accuracies.append(np.mean(fold_accs))

best_k = k_values[np.argmax(k_accuracies)]
print(f'Best K = {best_k} with CV Accuracy = {max(k_accuracies):.4f}')

# Plot K vs Accuracy
plt.figure(figsize=(9, 4))
plt.plot(k_values, k_accuracies, marker='o', color='steelblue', linewidth=2, markersize=5)
plt.axvline(x=best_k, color='red', linestyle='--', label=f'Best K={best_k}')
plt.xlabel('K (Number of Neighbors)', fontsize=11)
plt.ylabel('Average CV Accuracy', fontsize=11)
plt.title('KNN – K Value vs. Cross-Validation Accuracy', fontsize=13, fontweight='bold')
plt.legend(fontsize=10)
plt.xticks(k_values)
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

### 2.3 KNN – 5-Fold Cross-Validation with Best K

In [ ]:
knn_fold_results = []
knn_cms = []
knn_all_true = []
knn_all_pred = []

for fold_num, (train_idx, test_idx) in enumerate(skf.split(X_array, y_encoded)):
    X_train, X_test = X_array[train_idx], X_array[test_idx]
    y_train, y_test = y_encoded[train_idx], y_encoded[test_idx]

    scaler = StandardScaler()
    X_train_sc = scaler.fit_transform(X_train)
    X_test_sc  = scaler.transform(X_test)

    knn = KNeighborsClassifier(n_neighbors=best_k)
    knn.fit(X_train_sc, y_train)
    y_pred = knn.predict(X_test_sc)

    metrics = compute_metrics(y_test, y_pred)
    knn_fold_results.append(metrics)
    knn_cms.append(confusion_matrix(y_test, y_pred))
    knn_all_true.extend(y_test)
    knn_all_pred.extend(y_pred)

knn_results_df = show_results_table(knn_fold_results, 'KNN')

In [ ]:
plot_avg_metrics(knn_results_df, 'KNN')

In [ ]:
# Aggregate confusion matrix across all folds
knn_cm_total = confusion_matrix(knn_all_true, knn_all_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(knn_cm_total, annot=True, fmt='d', cmap='Blues',
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.xlabel('Predicted Label', fontsize=11)
plt.ylabel('True Label', fontsize=11)
plt.title(f'KNN (K={best_k}) – Aggregate Confusion Matrix (5-Fold)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 3. Classification Algorithm 2 – Support Vector Machine (SVM)

### 3.1 Why Scaling is Needed for SVM

SVM finds the **optimal hyperplane** that maximizes the margin between classes.  
The margin and support vectors depend on dot products (or kernel functions) between feature vectors.  
Features with larger ranges will dominate the margin calculation, so **StandardScaler** is applied to keep all features on equal footing.

### 3.2 Support Vectors and Decision Boundary

**Support vectors** are the training samples that lie closest to the decision boundary (hyperplane).  
They are the only samples that actually define where the boundary is placed.  
The SVM objective is to maximize the distance (margin) between these support vectors from each class.  
A wider margin generally means better generalization to unseen data.

### 3.3 Test Different Kernels

In [ ]:
# Compare linear, RBF, and polynomial kernels
kernels = ['linear', 'rbf', 'poly']
kernel_accs = {}

for kernel in kernels:
    fold_accs = []
    for train_idx, test_idx in skf.split(X_array, y_encoded):
        X_train, X_test = X_array[train_idx], X_array[test_idx]
        y_train, y_test = y_encoded[train_idx], y_encoded[test_idx]

        scaler = StandardScaler()
        X_train_sc = scaler.fit_transform(X_train)
        X_test_sc  = scaler.transform(X_test)

        svm = SVC(kernel=kernel, C=1.0, random_state=SEED)
        svm.fit(X_train_sc, y_train)
        fold_accs.append(accuracy_score(y_test, svm.predict(X_test_sc)))

    kernel_accs[kernel] = np.mean(fold_accs)
    print(f'Kernel: {kernel:8s} | Average CV Accuracy: {np.mean(fold_accs):.4f}')

best_kernel = max(kernel_accs, key=kernel_accs.get)
print(f'\nBest kernel: {best_kernel} (Accuracy = {kernel_accs[best_kernel]:.4f})')

# Bar chart for kernel comparison
plt.figure(figsize=(6, 4))
plt.bar(kernel_accs.keys(), kernel_accs.values(),
        color=['steelblue', 'tomato', 'mediumseagreen'], edgecolor='black', width=0.4)
for k, v in kernel_accs.items():
    plt.text(list(kernel_accs.keys()).index(k), v + 0.003, f'{v:.4f}', ha='center', fontsize=10)
plt.ylim(0, 1.05)
plt.xlabel('Kernel', fontsize=11)
plt.ylabel('Average CV Accuracy', fontsize=11)
plt.title('SVM – Kernel Comparison (5-Fold CV)', fontsize=12, fontweight='bold')
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

### 3.4 SVM – 5-Fold Cross-Validation with Best Kernel

In [ ]:
svm_fold_results = []
svm_all_true = []
svm_all_pred = []

for fold_num, (train_idx, test_idx) in enumerate(skf.split(X_array, y_encoded)):
    X_train, X_test = X_array[train_idx], X_array[test_idx]
    y_train, y_test = y_encoded[train_idx], y_encoded[test_idx]

    scaler = StandardScaler()
    X_train_sc = scaler.fit_transform(X_train)
    X_test_sc  = scaler.transform(X_test)

    svm_model = SVC(kernel=best_kernel, C=1.0, random_state=SEED)
    svm_model.fit(X_train_sc, y_train)
    y_pred = svm_model.predict(X_test_sc)

    metrics = compute_metrics(y_test, y_pred)
    svm_fold_results.append(metrics)
    svm_all_true.extend(y_test)
    svm_all_pred.extend(y_pred)

svm_results_df = show_results_table(svm_fold_results, 'SVM')

In [ ]:
plot_avg_metrics(svm_results_df, f'SVM ({best_kernel} kernel)')

In [ ]:
# Aggregate confusion matrix
svm_cm_total = confusion_matrix(svm_all_true, svm_all_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(svm_cm_total, annot=True, fmt='d', cmap='Oranges',
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.xlabel('Predicted Label', fontsize=11)
plt.ylabel('True Label', fontsize=11)
plt.title(f'SVM ({best_kernel}) – Aggregate Confusion Matrix (5-Fold)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

### 3.5 Support Vectors Visualization (2D Projection)

In [ ]:
# Train SVM on full data (scaled) and visualize support vectors
# using the two most informative features for illustration
scaler_full = StandardScaler()
X_full_sc = scaler_full.fit_transform(X_array)

# Use SystolicBP (index 1) and BS (index 3) for 2D visualization
feat_idx = [1, 3]
X_2d = X_full_sc[:, feat_idx]

svm_2d = SVC(kernel=best_kernel, C=1.0, random_state=SEED)
svm_2d.fit(X_2d, y_encoded)

fig, ax = plt.subplots(figsize=(8, 5))

class_colors = ['green', 'orange', 'red']
for i, label in enumerate(le.classes_):
    idx = y_encoded == i
    ax.scatter(X_2d[idx, 0], X_2d[idx, 1],
               c=class_colors[i], label=label, alpha=0.5, s=20, edgecolors='none')

# Highlight support vectors
sv = svm_2d.support_vectors_
ax.scatter(sv[:, 0], sv[:, 1], s=80, facecolors='none',
           edgecolors='black', linewidths=1.2, label='Support Vectors')

ax.set_xlabel('SystolicBP (scaled)', fontsize=11)
ax.set_ylabel('BS (scaled)', fontsize=11)
ax.set_title(f'SVM ({best_kernel}) – Support Vectors (2D projection)', fontsize=12, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

print(f'Total support vectors: {len(sv)}')

---
## 4. Classification Algorithm 3 – Deep Learning MLP

### 4.1 Network Architecture

We use **sklearn's MLPClassifier** (a fully-connected feedforward neural network).

| Layer | Neurons | Activation |
|-------|---------|------------|
| Input | 6 (one per feature) | – |
| Hidden 1 | 64 | ReLU |
| Hidden 2 | 32 | ReLU |
| Output | 3 (one per class) | Softmax (implicit in log-loss) |

**Why ReLU?** It avoids the vanishing gradient problem and allows the network to learn non-linear boundaries efficiently.  
**Batch size:** The `batch_size='auto'` setting in sklearn uses min(200, n_samples), which balances gradient stability and training speed.  
**Learning rate:** Set to 0.001 (adaptive), a standard starting point for Adam-style optimizers.  
**max_iter (epochs):** 500 iterations give the network enough time to converge without overfitting.

### 4.2 MLP – 5-Fold Cross-Validation

In [ ]:
mlp_fold_results = []
mlp_all_true = []
mlp_all_pred = []

for fold_num, (train_idx, test_idx) in enumerate(skf.split(X_array, y_encoded)):
    X_train, X_test = X_array[train_idx], X_array[test_idx]
    y_train, y_test = y_encoded[train_idx], y_encoded[test_idx]

    # Scale inside the fold
    scaler = StandardScaler()
    X_train_sc = scaler.fit_transform(X_train)
    X_test_sc  = scaler.transform(X_test)

    # MLP with two hidden layers: 64 and 32 neurons
    mlp = MLPClassifier(
        hidden_layer_sizes=(64, 32),
        activation='relu',
        solver='adam',
        learning_rate_init=0.001,
        batch_size='auto',
        max_iter=500,
        random_state=SEED,
        early_stopping=True,
        validation_fraction=0.1,
        n_iter_no_change=20
    )
    mlp.fit(X_train_sc, y_train)
    y_pred = mlp.predict(X_test_sc)

    metrics = compute_metrics(y_test, y_pred)
    mlp_fold_results.append(metrics)
    mlp_all_true.extend(y_test)
    mlp_all_pred.extend(y_pred)

mlp_results_df = show_results_table(mlp_fold_results, 'MLP')

In [ ]:
plot_avg_metrics(mlp_results_df, 'Deep Learning MLP')

In [ ]:
# Aggregate confusion matrix
mlp_cm_total = confusion_matrix(mlp_all_true, mlp_all_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(mlp_cm_total, annot=True, fmt='d', cmap='Purples',
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.xlabel('Predicted Label', fontsize=11)
plt.ylabel('True Label', fontsize=11)
plt.title('MLP – Aggregate Confusion Matrix (5-Fold)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 5. Comparison and Conclusion
### 5.1 Comparison Table

In [ ]:
# Build comparison table: each row is one model, columns are average metrics
comparison = pd.DataFrame({
    'Model': ['KNN', 'SVM', 'MLP'],
    'Accuracy':    [knn_results_df['Accuracy'].mean(),
                    svm_results_df['Accuracy'].mean(),
                    mlp_results_df['Accuracy'].mean()],
    'Sensitivity': [knn_results_df['Sensitivity'].mean(),
                    svm_results_df['Sensitivity'].mean(),
                    mlp_results_df['Sensitivity'].mean()],
    'Specificity': [knn_results_df['Specificity'].mean(),
                    svm_results_df['Specificity'].mean(),
                    mlp_results_df['Specificity'].mean()],
    'Precision':   [knn_results_df['Precision'].mean(),
                    svm_results_df['Precision'].mean(),
                    mlp_results_df['Precision'].mean()],
    'Recall':      [knn_results_df['Recall'].mean(),
                    svm_results_df['Recall'].mean(),
                    mlp_results_df['Recall'].mean()],
    'F1-Score':    [knn_results_df['F1-Score'].mean(),
                    svm_results_df['F1-Score'].mean(),
                    mlp_results_df['F1-Score'].mean()],
})

comparison = comparison.set_index('Model')
print('=== Model Comparison (Average 5-Fold CV) ===')
print(comparison.round(4).to_string())

### 5.2 Comparison Bar Plot

In [ ]:
metrics_to_plot = comparison.columns.tolist()
x = np.arange(len(metrics_to_plot))
width = 0.25

fig, ax = plt.subplots(figsize=(12, 5))

for i, (model, row) in enumerate(comparison.iterrows()):
    bars = ax.bar(x + i * width, row.values, width, label=model, edgecolor='black')
    for bar, val in zip(bars, row.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.004,
                f'{val:.3f}', ha='center', va='bottom', fontsize=7.5)

ax.set_xticks(x + width)
ax.set_xticklabels(metrics_to_plot, fontsize=10)
ax.set_ylabel('Average Score', fontsize=11)
ax.set_ylim(0, 1.12)
ax.set_title('KNN vs SVM vs MLP – Model Comparison (5-Fold CV)', fontsize=13, fontweight='bold')
ax.legend(title='Model', fontsize=10)
ax.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

### 5.3 Strengths and Weaknesses Discussion

| Model | Strengths | Weaknesses |
|-------|-----------|------------|
| **KNN** | Simple, no training phase, intuitive | Slow at prediction (stores all data), sensitive to irrelevant features |
| **SVM** | Effective in high dimensions, robust to outliers | Slow on large datasets, kernel tuning required |
| **MLP** | Learns complex non-linear patterns, scalable | Requires more data and tuning, less interpretable |

### 5.4 Conclusion

Classification algorithms can play a critical role in maternal healthcare.  
Early and accurate prediction of maternal risk levels—low, mid, or high—can help healthcare providers prioritize interventions and potentially prevent serious complications during pregnancy.  

In this project, all three classifiers (KNN, SVM, MLP) were evaluated on the UCI Maternal Health Risk dataset using 5-fold stratified cross-validation. The results show that SVM and MLP generally outperform KNN, with the non-linear kernel and the neural network both capturing complex relationships between vital signs and risk levels.  

Real-world deployment of such a model in IoT-connected clinics could enable automated triage, reduce physician workload, and improve outcomes in resource-limited settings.

---
## 6. Video Demonstration

**Demo Video Link:** [Insert YouTube / Google Drive link here]

The video demonstrates:
1. Opening and running this notebook in Google Colab
2. Dataset loading and EDA outputs
3. KNN K-selection plot and confusion matrix
4. SVM kernel comparison and support vector visualization
5. MLP architecture and training results
6. Final comparison table and bar chart